In [4]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI

In [5]:
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [6]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

print("LangChain과 Neo4j 연결 성공!")

LangChain과 Neo4j 연결 성공!


In [7]:
graph.refresh_schema() # 스키마 새로고침

print(graph.schema)

Node properties:
Student {age: INTEGER, name: STRING, student_id: INTEGER}
Course {name: STRING, course_id: INTEGER, level: STRING, duration: INTEGER}
Instructor {name: STRING, career: INTEGER, instructor_id: INTEGER, carrer: INTEGER}
Category {name: STRING, category_id: INTEGER}
Relationship properties:
ENROLLED_IN {score: INTEGER, enrolled_at: DATE}
The relationships:
(:Student)-[:ENROLLED_IN]->(:Course)
(:Course)-[:BELONGS_TO]->(:Category)
(:Instructor)-[:TEACHES]->(:Course)


In [8]:
query = """
MATCH
    (student:Student)
    -[:ENROLLED_IN]->
    (course:Course)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    student.student_id AS student_id,
    course.course_id AS course_id

ORDER BY student_id, course_id
"""

result = graph.query(query) # 쿼리를 받아 결과를 딕셔너리 list(dict)로 반환

result


[{'student_name': '홍길동',
  'course_name': 'python',
  'student_id': 1,
  'course_id': 101},
 {'student_name': '홍길동',
  'course_name': 'Data Analysis',
  'student_id': 1,
  'course_id': 104},
 {'student_name': '김영희',
  'course_name': 'Database',
  'student_id': 2,
  'course_id': 102},
 {'student_name': '김영희',
  'course_name': 'Machine Learning',
  'student_id': 2,
  'course_id': 103},
 {'student_name': '이민수',
  'course_name': 'python',
  'student_id': 3,
  'course_id': 101},
 {'student_name': '박서연',
  'course_name': 'Machine Learning',
  'student_id': 4,
  'course_id': 103},
 {'student_name': '박서연',
  'course_name': 'Deep Learning',
  'student_id': 4,
  'course_id': 105},
 {'student_name': '최준호',
  'course_name': 'Langchain',
  'student_id': 5,
  'course_id': 106}]

In [9]:
llm = ChatOpenAI(
    model = os.getenv('OPENAI_MODEL'),
    temperature = 0 # DB의 정보만 가져올 것이므로 창의성은 0 (결정론적인 답변 = 일관적)
)

In [10]:
# LLM과 Neo4j를 연결하여 자연어 질문을 Cypherfh qusghksgkrh ekqqusgksms cpdls
chain = GraphCypherQAChain.from_llm(
    llm = llm,      # Cypher 생성 및 최종 답변 llm
    graph = graph,  # 참조할 Neo4j 그래프 객체
    verbose = True, # 로그 출력
    validate_cypher = True, # 생성된 Cypher 검증
    return_intermediate_steps = True, # 중간과정 함께 반환
    top_k = 10, # 조회결과 10개
    allow_dangerous_requests = True # DB 쿼리 실행 위험성 확인
)

In [22]:
question = "Python 강의를 수강하는 학생을 알려줘"

response = chain.invoke({"query": question})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})
RETURN s;

Full Context:
[]

> Finished chain.


{'query': 'Python 강의를 수강하는 학생을 알려줘',
 'result': 'Python 강의를 수강하는 학생을 알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})\nRETURN s;\n"},
  {'context': []}]}

In [14]:
response.get('intermediate_steps') # 중간 과정 확인

[{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nWHERE c.name = 'Python'\nRETURN s.name;"},
 {'context': []}]

In [19]:
# 질문 답변 chain 함수
def ask_graph(question: str) -> dict:
    if not question.strip():
        raise ValueError("질문을 입력하셔야 합니다!!")

    response = chain.invoke({"query":question})

    print(f"[질문] {question}")
    print(f"[최종 답변] {response['result']}")

    # 중간과정 추가시
    for step in response.get('intermediate_steps',[]):
        if "query" in step:
            print(f"[생성된 Cypher] {step['query']}")
        
        if "context" in step:
            print(f"[조회 결과] {step['context']}")

    return response

In [20]:
ask_graph("홍길동이 수강한 강의와 담당 강사를 알려줘")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS course_name, i.name AS instructor_name;
Full Context:
[{'course_name': 'python', 'instructor_name': 'Capybara'}, {'course_name': 'Data Analysis', 'instructor_name': 'Alice'}]

> Finished chain.
[질문] 홍길동이 수강한 강의와 담당 강사를 알려줘
[최종 답변] 홍길동이 수강한 강의와 담당 강사에 대한 정보는 알 수 없습니다.
[생성된 Cypher] MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN c.name AS course_name, i.name AS instructor_name;
[조회 결과] [{'course_name': 'python', 'instructor_name': 'Capybara'}, {'course_name': 'Data Analysis', 'instructor_name': 'Alice'}]


{'query': '홍길동이 수강한 강의와 담당 강사를 알려줘',
 'result': '홍길동이 수강한 강의와 담당 강사에 대한 정보는 알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)\nRETURN c.name AS course_name, i.name AS instructor_name;"},
  {'context': [{'course_name': 'python', 'instructor_name': 'Capybara'},
    {'course_name': 'Data Analysis', 'instructor_name': 'Alice'}]}]}

In [23]:
ask_graph('홍길동과 같은 강의를 수강한 다른 학생을 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (target:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other <> target
RETURN DISTINCT other.name AS student_name;
Full Context:
[{'student_name': '이민수'}]

> Finished chain.
[질문] 홍길동과 같은 강의를 수강한 다른 학생을 알려줘
[최종 답변] 알 수 없습니다.
[생성된 Cypher] MATCH (target:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other <> target
RETURN DISTINCT other.name AS student_name;
[조회 결과] [{'student_name': '이민수'}]


{'query': '홍길동과 같은 강의를 수강한 다른 학생을 알려줘',
 'result': '알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (target:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)\nWHERE other <> target\nRETURN DISTINCT other.name AS student_name;"},
  {'context': [{'student_name': '이민수'}]}]}

In [24]:
ask_graph('capybara 강사가 담당하는 강의를 수강하는 학생들을 알려줘')



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (i:Instructor {name: 'capybara'})-[:TEACHES]->(c:Course)<-[:ENROLLED_IN]-(s:Student)
RETURN DISTINCT s.name AS student_name;
Full Context:
[]

> Finished chain.
[질문] capybara 강사가 담당하는 강의를 수강하는 학생들을 알려줘
[최종 답변] 정보가 없어 capybara 강사가 담당하는 강의를 수강하는 학생들을 알 수 없습니다.
[생성된 Cypher] MATCH (i:Instructor {name: 'capybara'})-[:TEACHES]->(c:Course)<-[:ENROLLED_IN]-(s:Student)
RETURN DISTINCT s.name AS student_name;
[조회 결과] []


{'query': 'capybara 강사가 담당하는 강의를 수강하는 학생들을 알려줘',
 'result': '정보가 없어 capybara 강사가 담당하는 강의를 수강하는 학생들을 알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (i:Instructor {name: 'capybara'})-[:TEACHES]->(c:Course)<-[:ENROLLED_IN]-(s:Student)\nRETURN DISTINCT s.name AS student_name;"},
  {'context': []}]}